# NILM Deep Learning Disaggregator Pipeline


### Imports


In [ ]:
# %%
import os
import json
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import optuna
from scipy.fft import fft, fftfreq
from scipy.stats import skew, kurtosis
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# %%


### # 1. CONSTANTS & DATA LOADING


In [ ]:
DATA_DIR = './Data'
META_2014 = os.path.join(DATA_DIR, 'meta_2014.json')
META_2017 = os.path.join(DATA_DIR, 'meta_2017.json')
CSV_DIR = os.path.join(DATA_DIR, '2017')

FS = 30000  # 30 kHz
WINDOW_SIZE = FS  # 1-second window

# ADD OR REMOVE ANY NUMBER OF APPLIANCES HERE
TARGET_APPLIANCES = ['Microwave', 'Heater', 'Fridge', 'Air Conditioner', 'Washing Machine', 'Vacuum']

def parse_metadata():
    all_meta = []
    if os.path.exists(META_2014):
        with open(META_2014, 'r') as f:
            all_meta.extend(json.load(f))
    if os.path.exists(META_2017):
        with open(META_2017, 'r') as f:
            all_meta.extend(json.load(f))
            
    appliance_ids = defaultdict(list)
    for item in all_meta:
        aid = item['id']
        app_type = item['meta']['appliance']['type']
        if os.path.exists(os.path.join(CSV_DIR, f"{aid}.csv")):
            appliance_ids[app_type].append(aid)
            
    return appliance_ids

def load_windows(ids, max_windows=None):
    V_windows, I_windows, P_windows = [], [], []
    for aid in tqdm(ids, desc="Loading data"):
        filepath = os.path.join(CSV_DIR, f"{aid}.csv")
        try:
            df = pd.read_csv(filepath, header=None, names=['current', 'voltage'])
            v = df['voltage'].values
            i = df['current'].values
            
            num_windows = len(v) // WINDOW_SIZE
            for w in range(num_windows):
                start = w * WINDOW_SIZE
                end = start + WINDOW_SIZE
                v_win, i_win = v[start:end], i[start:end]
                p_win = np.mean(v_win * i_win)
                
                V_windows.append(v_win)
                I_windows.append(i_win)
                P_windows.append(abs(p_win))
                
                if max_windows and len(V_windows) >= max_windows:
                    return np.array(V_windows), np.array(I_windows), np.array(P_windows)
        except Exception:
            continue
    return np.array(V_windows), np.array(I_windows), np.array(P_windows)

def augment_signal(v, i, snr_db=40):
    scale = np.random.uniform(0.95, 1.05)
    i_aug = i * scale
    i_power = np.mean(i_aug**2)
    noise_power = i_power / (10 ** (snr_db / 10))
    noise = np.random.normal(0, np.sqrt(noise_power), len(i_aug))
    i_aug = i_aug + noise
    shift = np.random.randint(-FS//60, FS//60)
    i_aug = np.roll(i_aug, shift)
    return v, i_aug

def synthesize_aggregates(appliance_data_list, num_samples=1000):
    V_agg = np.zeros((num_samples, WINDOW_SIZE))
    I_agg = np.zeros((num_samples, WINDOW_SIZE))
    P_targets = np.zeros((num_samples, len(appliance_data_list)))
    
    lengths = [len(app[0]) if app[0] is not None else 0 for app in appliance_data_list]
    
    for k in tqdm(range(num_samples), desc="Synthesizing aggregates"):
        indices = [np.random.randint(0, n) if n > 0 else 0 for n in lengths]
        
        V_agg_win = np.zeros(WINDOW_SIZE)
        I_agg_win = np.zeros(WINDOW_SIZE)
        P_target = np.zeros(len(appliance_data_list))
        
        # Ensure voltage is taken from the first available appliance
        for i, app_data in enumerate(appliance_data_list):
            if app_data[0] is not None and len(app_data[0]) > 0:
                V_agg_win = app_data[0][indices[i]].copy()
                break
                
        # Sparsely activate appliances (simulating a real sparse household load)
        for i, app_data in enumerate(appliance_data_list):
            if np.random.rand() < 0.3:  # 30% probability of being ON
                idx = indices[i]
                if app_data[0] is not None and len(app_data[0]) > 0:
                    I_agg_win += app_data[1][idx]
                    P_target[i] = app_data[2][idx]
            
        V_agg[k] = V_agg_win
        I_agg[k] = I_agg_win
        P_targets[k] = P_target
        
    return V_agg, I_agg, P_targets

class NILMDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]

def get_dataloaders(X, Y, batch_size=32, seq_length=10):
    X_seq, Y_seq = [], []
    for i in range(len(X) - seq_length + 1):
        X_seq.append(X[i : i+seq_length])
        Y_seq.append(Y[i + seq_length - 1])
        
    X_seq, Y_seq = np.array(X_seq), np.array(Y_seq)
    
    n_samples = len(X_seq)
    train_end = int(0.7 * n_samples)
    val_end = int(0.85 * n_samples)
    
    X_train, y_train = X_seq[:train_end], Y_seq[:train_end]
    X_val, y_val = X_seq[train_end:val_end], Y_seq[train_end:val_end]
    X_test, y_test = X_seq[val_end:], Y_seq[val_end:]
    
    train_loader = DataLoader(NILMDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(NILMDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(NILMDataset(X_test, y_test), batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader, y_train, y_val, y_test

# %%


### # 2. FEATURE EXTRACTION


In [ ]:
def extract_window_features(v, i, fs=30000):
    N = len(v)
    p_inst = v * i
    P = np.mean(p_inst)
    Vrms = np.sqrt(np.mean(v**2))
    Irms = np.sqrt(np.mean(i**2))
    S = Vrms * Irms
    Q = np.sqrt(max(S**2 - P**2, 0))
    PF = P / S if S > 0 else 0
    
    v_fft = np.fft.rfft(v) * 2.0 / N
    i_fft = np.fft.rfft(i) * 2.0 / N
    
    i_mag = np.abs(i_fft)
    thd = 0
    if i_mag[60] > 0:
        thd = np.sqrt(np.sum(i_mag[61:]**2)) / i_mag[60]
        
    base_feats = [Vrms, Irms, P, Q, S, PF, thd]
    
    harmonic_feats = []
    # Extract up to 60 harmonics (60 Hz to 3600 Hz)
    for h in range(1, 61):
        idx = h * 60
        if idx < len(i_fft):
            i_val = i_fft[idx]
            v_val = v_fft[idx]
            harmonic_feats.extend([i_val.real, i_val.imag, v_val.real, v_val.imag])
        else:
            harmonic_feats.extend([0, 0, 0, 0])
            
    features = base_feats + harmonic_feats
    return np.array(features)

def extract_dataset_features(V_windows, I_windows):
    features_list = []
    for w in tqdm(range(len(V_windows)), desc="Extracting features"):
        features_list.append(extract_window_features(V_windows[w], I_windows[w]))
    return np.nan_to_num(np.array(features_list))

# %%


### # 3. MODEL ARCHITECTURE


In [ ]:
class NNAN_Model(nn.Module):
    def __init__(self, num_features=18, seq_length=10, hidden_size=128, num_targets=2, dropout_rate=0.3):
        super(NNAN_Model, self).__init__()
        self.conv1 = nn.Conv1d(num_features, 128, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(128)
        self.conv2 = nn.Conv1d(128, 256, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(256)
        self.conv3 = nn.Conv1d(256, 256, 3, padding=1)
        self.bn3 = nn.BatchNorm1d(256)
        
        self.res_conv = nn.Conv1d(num_features, 256, 1)
        self.dropout = nn.Dropout(dropout_rate)
        
        self.lstm = nn.LSTM(256, hidden_size*2, num_layers=3, batch_first=True, bidirectional=True,
                            dropout=dropout_rate if dropout_rate > 0 else 0)
        
        self.attention = nn.MultiheadAttention(hidden_size * 4, num_heads=8, batch_first=True, dropout=dropout_rate)
        self.layer_norm = nn.LayerNorm(hidden_size * 4)
        
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear((hidden_size * 4) * seq_length, 1024)
        self.fc2 = nn.Linear(1024, 256)
        self.fc3 = nn.Linear(256, num_targets)
        
    def forward(self, x):
        x = x.transpose(1, 2)
        res = self.res_conv(x)
        x = self.dropout(F.relu(self.bn1(self.conv1(x))))
        x = self.dropout(F.relu(self.bn2(self.conv2(x))))
        x = self.bn3(self.conv3(x))
        x = F.relu(x + res)
        x = x.transpose(1, 2)
        
        lstm_out, _ = self.lstm(x)
        attn_out, attn_weights = self.attention(lstm_out, lstm_out, lstm_out)
        x = self.layer_norm(lstm_out + attn_out)
        
        x = self.flatten(x)
        x = self.dropout(F.relu(self.fc1(x)))
        x = F.relu(self.fc2(x))
        out = self.fc3(x)
        return out, attn_weights

# %%


### # 4. TRAINING & EVALUATION


In [ ]:
def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train_epoch(model, dataloader, criterion, optimizer, device, weights=None):
    model.train()
    running_loss = 0.0
    for X_batch, Y_batch in dataloader:
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
        optimizer.zero_grad()
        outputs, _ = model(X_batch)
        loss_all = criterion(outputs, Y_batch)
        loss = (loss_all * weights).mean() if weights is not None else loss_all.mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(dataloader)

def validate(model, dataloader, criterion, device, weights=None):
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, Y_batch in dataloader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            outputs, _ = model(X_batch)
            loss_all = criterion(outputs, Y_batch)
            loss = (loss_all * weights).mean() if weights is not None else loss_all.mean()
            val_loss += loss.item()
    return val_loss / len(dataloader)

def run_optuna_study(X, Y, num_targets, Y_std, n_trials=10):
    device = get_device()
    num_features = X.shape[1]
    
    def objective(trial):
        seq_length = 1  # Fixed to 1 since synthetic dataset lacks temporal sequence continuity
        lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        hidden_size = trial.suggest_categorical('hidden_size', [64, 128])
        batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
        dropout = trial.suggest_float('dropout', 0.1, 0.4)
        
        if len(X) <= seq_length: return float('inf')
        
        train_loader, val_loader, _, _, _, _ = get_dataloaders(X, Y, batch_size, seq_length)
        model = NNAN_Model(num_features, seq_length, hidden_size, num_targets=num_targets, dropout_rate=dropout).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        criterion = nn.L1Loss(reduction='none')
        weights = torch.tensor(Y_std, dtype=torch.float32, device=device)
        
        best_val_loss = float('inf')
        for epoch in range(15):
            train_epoch(model, train_loader, criterion, optimizer, device, weights)
            val_loss = validate(model, val_loader, criterion, device, weights)
            if val_loss < best_val_loss: best_val_loss = val_loss
            trial.report(val_loss, epoch)
            if trial.should_prune(): raise optuna.exceptions.TrialPruned()
        return best_val_loss

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_trial.params

def train_best_model(X, Y, params, num_targets, Y_std, epochs=150, save_dir='models'):
    device = get_device()
    os.makedirs(save_dir, exist_ok=True)
    num_features = X.shape[1]
    
    seq_length = params.get('seq_length', 1)
    lr, hidden_size = params['lr'], params['hidden_size']
    batch_size, dropout = params['batch_size'], params['dropout']
    num_features = X.shape[1]
    
    train_loader, val_loader, test_loader, _, _, y_test = get_dataloaders(X, Y, batch_size, seq_length)
    model = NNAN_Model(num_features, seq_length, hidden_size, num_targets=num_targets, dropout_rate=dropout).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    criterion = nn.L1Loss(reduction='none')
    weights = torch.tensor(Y_std, dtype=torch.float32, device=device)
    
    best_val_loss = float('inf')
    patience_counter, patience = 0, 30
    train_losses, val_losses = [], []
    
    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device, weights)
        val_loss = validate(model, val_loader, criterion, device, weights)
        scheduler.step(val_loss)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
            
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), os.path.join(save_dir, 'best_model.pth'))
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience: break
            
    model.load_state_dict(torch.load(os.path.join(save_dir, 'best_model.pth'), weights_only=True))
    return model, test_loader, train_losses, val_losses

def evaluate_model(model, test_loader, device, y_scaler_mean, y_scaler_std, appliance_names, save_dir='results'):
    os.makedirs(save_dir, exist_ok=True)
    model.eval()
    
    predictions, ground_truths, attentions = [], [], []
    with torch.no_grad():
        for X_batch, Y_batch in test_loader:
            X_batch = X_batch.to(device)
            out, attn = model(X_batch)
            predictions.append(out.cpu().numpy())
            ground_truths.append(Y_batch.numpy())
            attentions.append(attn.cpu().numpy())
            
    predictions = np.vstack(predictions)
    ground_truths = np.vstack(ground_truths)
    attentions = np.concatenate(attentions, axis=0)
    
    pred_w = np.maximum(predictions * y_scaler_std + y_scaler_mean, 0)
    gt_w = ground_truths * y_scaler_std + y_scaler_mean
    
    agg_w = np.sum(gt_w, axis=1)
    num_apps = len(appliance_names)
    
    print("\n--- Test Metrics ---")
    for i, name in enumerate(appliance_names):
        y_true, y_pred = gt_w[:, i], pred_w[:, i]
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)
        nde = np.sum((y_pred - y_true)**2) / np.sum(y_true**2) if np.sum(y_true**2) > 0 else 0
        print(f"{name}:\n  MAE:  {mae:.2f} W\n  RMSE: {rmse:.2f} W\n  R2:   {r2:.4f}\n  NDE:  {nde:.4f}")
        
    # --- PLOTS ---
    plot_len = min(100, len(pred_w))
    
    # 1. Aggregated vs Components (Separate)
    fig, axs = plt.subplots(num_apps, 1, figsize=(14, 5 * num_apps))
    if num_apps == 1: axs = [axs]
    
    for i, name in enumerate(appliance_names):
        axs[i].plot(agg_w[:plot_len], label='Aggregate Power (Ground Truth Sum)', color='black', linewidth=2)
        axs[i].plot(gt_w[:plot_len, i], label=f'GT {name}', alpha=0.6)
        axs[i].plot(pred_w[:plot_len, i], label=f'Pred {name}', linestyle='--')
        axs[i].set_title(f'Disaggregation: Aggregate vs {name} (First {plot_len} windows)')
        axs[i].set_ylabel('Power (W)')
        axs[i].legend(loc='upper right')
    axs[-1].set_xlabel('Time Window')
    
    plt.tight_layout()
    plt.show()
    plt.close()
    
    # 2. Individual Power Traces
    fig, axs = plt.subplots(num_apps, 1, figsize=(14, 4 * num_apps))
    if num_apps == 1: axs = [axs]
    for i, name in enumerate(appliance_names):
        axs[i].plot(gt_w[:plot_len, i], label='Ground Truth')
        axs[i].plot(pred_w[:plot_len, i], label='Prediction', linestyle='--')
        axs[i].set_title(f'{name} Power Disaggregation')
        axs[i].set_ylabel('Power (W)')
        axs[i].legend()
    axs[-1].set_xlabel('Time Window')
    plt.tight_layout()
    plt.show()
    plt.close()
    
    # 3. Scatter Plots
    cols = min(3, num_apps)
    rows = (num_apps + cols - 1) // cols
    fig, axs = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    axs_flat = np.array(axs).flatten() if num_apps > 1 else [axs]
    for i, name in enumerate(appliance_names):
        axs_flat[i].scatter(gt_w[:, i], pred_w[:, i], alpha=0.5, s=10)
        max_val = max(np.max(gt_w[:, i]), np.max(pred_w[:, i]))
        axs_flat[i].plot([0, max_val], [0, max_val], 'r--', label='Perfect Prediction')
        axs_flat[i].set_title(f'{name} - Prediction vs Actual')
        axs_flat[i].set_xlabel('Actual Power (W)')
        axs_flat[i].set_ylabel('Predicted Power (W)')
        axs_flat[i].legend()
    if num_apps > 1:
        for i in range(num_apps, len(axs_flat)):
            fig.delaxes(axs_flat[i])
    plt.tight_layout()
    plt.show()
    plt.close()
    
    # 4. Attention Heatmap
    if attentions.shape[0] > 0:
        sample_attn = attentions[0]
        plt.figure(figsize=(8, 6))
        sns.heatmap(sample_attn, cmap='viridis', annot=False)
        plt.title('Self-Attention Weights (Sample 0)')
        plt.xlabel('Sequence Step')
        plt.ylabel('Sequence Step')
        plt.tight_layout()
        plt.show()
        plt.close()

# %%


### # 5. EXPLORATORY DATA ANALYSIS (EDA)


In [ ]:
def run_eda(appliance_data_dict, save_dir='results'):
    os.makedirs(save_dir, exist_ok=True)
    print("\nGenerating EDA Plots...")
    
    num_apps = len(appliance_data_dict)
    samples = 1000
    
    # 1. Waveforms
    fig, axs = plt.subplots(num_apps, 1, figsize=(14, 4 * num_apps))
    if num_apps == 1: axs = [axs]
    
    for ax, (name, (V, I, P)) in zip(axs, appliance_data_dict.items()):
        v, i = V[0], I[0]
        ax.plot(v[:samples], color='orange', label='Voltage')
        ax2 = ax.twinx()
        ax2.plot(i[:samples], color='blue', label='Current')
        ax.set_title(f"{name} - Time Domain (2 Cycles)")
        ax.legend(loc='upper right')
        ax2.legend(loc='lower right')
        
    plt.tight_layout()
    plt.show()
    plt.close()
    
    # 2. V-I Trajectory
    fig, axs = plt.subplots(1, num_apps, figsize=(6 * num_apps, 5))
    if num_apps == 1: axs = [axs]
    for ax, (name, (V, I, P)) in zip(axs, appliance_data_dict.items()):
        v, i = V[0], I[0]
        ax.plot(v[:samples], i[:samples], color='blue')
        ax.set_title(f"{name} - V-I Trajectory")
        ax.set_xlabel("Voltage")
        ax.set_ylabel("Current")
        
    plt.tight_layout()
    plt.show()
    plt.close()
    
    # 3. Harmonics Comparison
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(7)
    width = 0.8 / num_apps
    
    for idx, (name, (V, I, P)) in enumerate(appliance_data_dict.items()):
        f = extract_window_features(V[0], I[0])
        harmonics = f[-7:]
        ax.bar(x + idx * width - 0.4 + width/2, harmonics, width, label=name)
        
    ax.set_title("Current Harmonics Comparison")
    ax.set_xticks(x)
    ax.set_xticklabels(['60Hz', '180Hz', '300Hz', '420Hz', '540Hz', '660Hz', '780Hz'])
    ax.set_ylabel("Normalized Amplitude")
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    plt.close()
    print("EDA plots saved to 'results' folder.")


# %%


### # 6. MAIN EXECUTION


In [ ]:
def main():
    print("=== NILM Deep Learning Disaggregator ===")
    print(f"Device: {get_device()}")
    
    print("\n[1/6] Parsing metadata...")
    ids_by_type = parse_metadata()
    
    appliance_data_list = []
    appliance_data_dict = {}
    
    print("\n[2/6] Loading windows from CSV...")
    for app_name in TARGET_APPLIANCES:
        app_ids = ids_by_type.get(app_name, [])
        print(f"Loading '{app_name}' (Found {len(app_ids)} files)...")
        V, I, P = load_windows(app_ids)
        print(f"Loaded {len(V)} windows for {app_name}.")
        if len(V) > 0:
            appliance_data_list.append((V, I, P))
            appliance_data_dict[app_name] = (V, I, P)
            
    if len(appliance_data_dict) == len(TARGET_APPLIANCES) and len(TARGET_APPLIANCES) > 0:
        run_eda(appliance_data_dict)
    else:
        print("Skipping EDA since some appliances have no data.")
        
    if len(appliance_data_list) == 0:
        print("No appliance data loaded. Exiting.")
        return
        
    print("\n[3/6] Synthesizing aggregate signals and augmenting...")
    num_synthetic = 20000
    V_agg, I_agg, P_targets = synthesize_aggregates(appliance_data_list, num_samples=num_synthetic)
    print(f"Created {len(V_agg)} aggregate windows.")
    
    if len(V_agg) == 0:
        print("Not enough data to train. Exiting.")
        return
    
    print("\n[4/6] Extracting electrical and spectral features...")
    X_features = extract_dataset_features(V_agg, I_agg)
    print(f"Feature matrix shape: {X_features.shape}")
    
    print("Normalizing features and targets...")
    X_mean, X_std = np.mean(X_features, axis=0), np.std(X_features, axis=0) + 1e-8
    X_norm = np.nan_to_num((X_features - X_mean) / X_std)
    
    Y_mean, Y_std = np.mean(P_targets, axis=0), np.std(P_targets, axis=0) + 1e-8
    Y_norm = np.nan_to_num((P_targets - Y_mean) / Y_std)
    
    print("\n[5/6] Tuning hyperparameters with Optuna...")
    num_targets = len(appliance_data_list)
    best_params = run_optuna_study(X_norm, Y_norm, num_targets=num_targets, Y_std=Y_std, n_trials=20)
    
    print("\n[6/6] Training best model and evaluating...")
    best_model, test_loader, train_losses, val_losses = train_best_model(
        X_norm, Y_norm, best_params, num_targets=num_targets, Y_std=Y_std, epochs=150
    )
    
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Huber Loss')
    plt.legend()
    plt.show()
    plt.close()
    
    evaluate_model(
        best_model, 
        test_loader, 
        get_device(), 
        y_scaler_mean=Y_mean, 
        y_scaler_std=Y_std, 
        appliance_names=list(appliance_data_dict.keys())
    )
    print("\nPipeline complete! Check 'results' folder for all plots.")

# %%
if __name__ == "__main__":
    main()
